# Baseline Clásico: SVM con características HOG + HSV

Este notebook implementa el baseline clásico para la clasificación de 4 defectos en papas. Utilizaremos un modelo de Support Vector Machine (SVM) alimentado por una combinación de características de textura (HOG) y color (HSV).

## 1. Importación de Librerías
Importamos las librerías necesarias para el procesamiento de imágenes, manipulación de arreglos y modelado de Machine Learning.

In [3]:
import os
import cv2
import numpy as np
import glob
from skimage.feature import hog
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 2. Extracción de Características (HOG + HSV)
Definimos una función que lee una imagen, la redimensiona y extrae:
- **Descriptores HOG**: capturan la textura y forma.
- **Histogramas HSV**: capturan la distribución del color.
Finalmente, concatena ambos en un único vector 1D.

In [ ]:
def extract_features(image_path, target_size=(128, 128)):
    img = cv2.imread(image_path)
    if img is None:
        print(f'Advertencia: No se pudo leer {image_path}')
        return None
        
    img_resized = cv2.resize(img, target_size)
    
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    hog_features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys',
                       visualize=False, feature_vector=True)
    
    hsv_img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
    hist_h = cv2.calcHist([hsv_img], [0], None, [16], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv_img], [1], None, [16], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv_img], [2], None, [16], [0, 256]).flatten()
    
    cv2.normalize(hist_h, hist_h)
    cv2.normalize(hist_s, hist_s)
    cv2.normalize(hist_v, hist_v)
    
    color_features = np.concatenate([hist_h, hist_s, hist_v])
    final_features = np.concatenate([hog_features, color_features])
    
    return final_features


## 3. Carga del Dataset
Recorremos el directorio ../data/, extraemos las características de cada imagen por clase y construimos nuestra matriz de características X y el vector de etiquetas y.

In [ ]:
def load_dataset(data_dir):
    X = []
    y = []
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    
    for label, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        image_paths = glob.glob(os.path.join(class_dir, '*.[jp][pn]*[g]'))
        
        print(f'Extrayendo características para clase {class_name}: {len(image_paths)} imágenes encontradas.')
        for img_path in image_paths:
            features = extract_features(img_path)
            if features is not None:
                X.append(features)
                y.append(label)
                
    return np.array(X), np.array(y), classes

data_dir = '../data/Potato_Dataset-20261002T071051Z-1-001/Potato_Dataset/'
X, y, class_names = load_dataset(data_dir)

print(f'\nDataset procesado.')
print(f'Dimensión de X (muestras, características): {X.shape}')
print(f'Clases identificadas: {class_names}')


## 4. Entrenamiento y Optimización
Creamos un pipeline que estandariza las características (StandardScaler) y luego entrena un SVM (SVC). Utilizamos GridSearchCV para encontrar los mejores valores de los hiperparámetros C y gamma. Finalmente evaluamos en un conjunto de prueba del 20%.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42, class_weight='balanced'))
])

param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 'auto', 0.01, 0.001],
    'svc__kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=2
)

print('Iniciando búsqueda de hiperparámetros...')
grid_search.fit(X_train, y_train)

print('\n--- Resultados de la Optimización ---')
print(f'Mejores parámetros encontrados: {grid_search.best_params_}')
print(f'Mejor F1-Macro (CV): {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

f1_final = f1_score(y_test, y_pred, average='macro')
print(f'\nF1-Macro Final en conjunto de prueba: {f1_final:.4f}\n')

print('Reporte de Clasificación Final:')
print(classification_report(y_test, y_pred, target_names=class_names))
